In [1]:
import sys
import os
import numpy as np

sys.path.append(os.path.abspath('..'))

from src.config import SolverConfig
from src.models.spacecraft import Spacecraft
from src.models.body import Body
from src.optimizer import Optimizer
from src.utils import cart2eq, cart2kep

from scripts.ephemerides import load_states

earth_path = "../src/horizons_results_earth_heliocentric_state_vector.txt"
mars_path = "../src/horizons_results_mars_heliocentric_state_vector.txt"

earth_loc, earth_dt = load_states(earth_path)
mars_loc, mars_dt = load_states(mars_path)

earth_loc = earth_loc*1e3
mars_loc = mars_loc*1e3

cfg = SolverConfig()
Earth = Body(name="earth",mu=3.986e14)
Mars = Body(name="mars",mu=4.283e13)
Sun = Body(name="sun",mu=1.327e20)

In [ ]:
from cyipopt import minimize_ipopt

# Your existing imports and classes (Psyche, Optimizer, cart2eq, etc.)
# Assume these are defined: earth_loc, mars_loc, Sun.mu, cfg

def objective(x, Psyche, cfg):
    """Objective: maximize final mass (minimize -final_mass)"""
    t0, tf = x  # departure days, transfer days
    
    # Initial/final conditions
    m0 = Psyche.m0
    r0v0 = earth_loc[int(t0)]
    y0 = np.hstack((r0v0, m0))
    rfvf = mars_loc[int(t0 + tf)]
    
    # Forward simulation
    fw_target = cart2eq(rfvf, Sun.mu)[0:6]
    fw_opt = Optimizer(cfg, Psyche, target_orbit=fw_target)
    fw_opt.controller.gains.nominal_throttle = 0.8
    
    fw_ts, fw_sol, *_ = fw_opt.simulate_forward(t0*86400, tf*86400, y0, mars_loc)
    
    # Penalty for non-convergence + mass consumption
    if not fw_opt.check_converge(fw_sol[0:6, -1]):
        return 1e6  # Huge penalty for failed convergence
    
    final_mass = fw_sol[6, -1]
    return -(final_mass - m0)  # Minimize mass consumed (maximize final mass)

def constraints(x):
    """Simple bounds as constraints"""
    t0, tf = x
    return [tf - 100, 1000 - tf]  # tf >= 100 days, tf <= 1000 days

def optimize_transfer(Psyche, cfg):
    """Main optimization using IPOPT"""
    
    # Bounds: t0 in [350,450], tf in [100,1000] days
    bounds = [(350, 450), (100, 1000)]
    
    # Initial guess (your original point)
    x0 = np.array([398.0, 692.0])
    
    # IPOPT options
    options = {
        'tol': 1e-6,
        'max_iter': 200,
        'print_level': 5,
        'mu_strategy': 'adaptive',
        'limited_memory_max_history': 10,
        'nlp_scaling_method': 'gradient-based'
    }
    
    # Run optimization
    result = minimize_ipopt(
        fun=objective,
        x0=x0,
        args=(Psyche, cfg),
        bounds=bounds,
        constraints={'type': 'ineq', 'fun': constraints},
        options=options,
        # ipopt_extras={'print_timing_statistics': 'yes'}
    )
    
    return result

# Run the optimization
print("Starting IPOPT optimization...")
Psyche = Spacecraft()
result = optimize_transfer(Psyche, cfg)

best_t0, best_tf = result.x
print(f"\n=== IPOPT SOLUTION ===")
print(f"Departure: {best_t0:.1f} days")
print(f"TOF: {best_tf:.1f} days") 
print(f"Arrival: {best_t0 + best_tf:.1f} days")
print(f"Status: {result.status}, Success: {result.success}")
print(f"Objective (Δm): {result.fun:.1f}kg")

# Re-run simulation to get full solution
if result.success:
    print("\nExtracting full solution...")
    m0 = Psyche.m0
    r0v0 = earth_loc[int(best_t0)]
    y0 = np.hstack((r0v0, m0))
    rfvf = mars_loc[int(best_t0 + best_tf)]
    
    fw_target = cart2eq(rfvf, Sun.mu)[0:6]
    fw_opt = Optimizer(cfg, Psyche, target_orbit=fw_target)
    fw_opt.controller.gains.nominal_throttle = 0.8
    
    best_fw_ts, best_sol, *_ = fw_opt.simulate_forward(
        best_t0*86400, best_tf*86400, y0, mars_loc
    )
    
    best_mf = best_sol[6, -1]
    print(f"Final mass: {best_mf:.1f}kg (Δm={m0-best_mf:.1f}kg)")
    print(f"Mass fraction: {best_mf/m0*100:.1f}%")
    
    # Save
    np.savez('best_transfer_ipopt.npz', 
             ts=best_fw_ts, sol=best_sol,
             departure=best_t0, tof=best_tf, final_mass=best_mf)
    print("Saved to 'best_transfer_ipopt.npz'")

Starting IPOPT optimization...
--- SWITCHING TO PHASE 2 (PHASING) ---


In [3]:
import numpy as np
from cyipopt import minimize_ipopt  # or scipy.optimize.minimize

def orbital_objective(x, Psyche, cfg, t0_fixed=398, tf_fixed=692):
    """
    Optimize Q-law weights: [w_r, w_v, w_a, w_m] → minimize orbital element errors
    x[0]: position weight (km)
    x[1]: velocity weight (km/s) 
    x[2]: acceleration weight (km/s²) - throttle priority
    x[3]: mass weight (kg) - fuel efficiency
    """
    # Fixed trajectory endpoints
    m0 = Psyche.m0
    r0v0 = earth_loc[int(t0_fixed)]
    y0 = np.hstack((r0v0, m0))
    rfvf = mars_loc[int(t0_fixed + tf_fixed)]
    
    # Target MEE (modified equinoctial elements)
    target_mee = cart2eq(rfvf, Sun.mu)
    
    # Decode weights (positive, reasonable bounds)
    w_a,w_f,w_g,w_h,w_k= x  # exp() ensures positivity
    
    # Run Q-law simulation with custom weights
    fw_target = target_mee[:6]
    fw_opt = Optimizer(cfg, Psyche, target_orbit=fw_target)
    
    # Set custom Q-law weights in controller
    fw_opt.controller.gains.W_oe = np.array([w_a,w_f,w_g,w_h,w_k,1.0])  # Adjust API as needed
    fw_opt.controller.gains.nominal_throttle = 0.8
    
    fw_ts, fw_sol, *_ = fw_opt.simulate_forward(
        t0_fixed*86400, tf_fixed*86400, y0, mars_loc
    )
    
    # Final state
    final_rv = fw_sol[0:6, -1]
    final_m = fw_sol[6, -1]
    
    # Check convergence first
    if not fw_opt.check_converge(final_rv):
        return 1e8  # Huge penalty
    
    # Orbital element errors (MEE: 6 elements)
    final_mee = np.array(cart2eq(final_rv, Sun.mu))
    mee_error = np.abs(final_mee - target_mee)
    
    # Weighted orbital error (primary objective)
    orbital_cost = (mee_error/np.array(fw_target)).sum()
    
    # Multi-objective: orbital + fuel
    fuel_cost = (m0 - final_m) / m0  # Normalized Δm
    
    total_cost = orbital_cost + fuel_cost
    
    print(f"  Weights: [{w_a:.1f},{w_f:.1f},{w_g:.1f},{w_h:.1f},{w_k:.1f},{w_L:.1f}] → "
          f"OrbErr={orbital_cost:.2f}, Δm={m0-final_m:.0f}kg")
    
    return total_cost

def optimize_q_weights(Psyche, cfg):
    """Full Q-law parameter optimization"""
    
    # Initial guess (log-scale for positivity)
    x0 = np.array([100.0,1.0,1.0,1.0,1.0])
    
    bounds = [
        (0.01, 100.0),
        (0.01, 100.0),
        (0.01, 100.0),
        (0.01, 100.0),
        (0.01, 100.0),
    ]
    
    options = {
        'tol': 1e-4,           # Slightly looser (noisy sims)
        'max_iter': 5,
        'print_level': 5,
        'acceptable_tol': 1e-3
    }
    
    print("Optimizing Q-law weights...")
    result = minimize_ipopt(
        fun=orbital_objective,
        x0=x0,
        args=(Psyche, cfg),
        bounds=bounds,
        options=options
    )
    
    # Decode optimal weights
    opt_weights = result.x
    print("\n=== OPTIMAL Q-LAW WEIGHTS ===")
    print(f"w_a: {opt_weights[0]:.1f}")
    print(f"w_f: {opt_weights[1]:.1f}")
    print(f"w_g: {opt_weights[2]:.1f}")
    print(f"w_h: {opt_weights[3]:.1f}")
    print(f"w_k: {opt_weights[4]:.1f}")
    print(f"Final cost: {result.fun:.2f}")
    
    return opt_weights, result

# Usage
Psyche = Spacecraft()
opt_weights, result = optimize_q_weights(Psyche, cfg)

# # Apply to final sim
# fw_opt.controller.weights = opt_weights
# # Run your full trajectory sim...

Optimizing Q-law weights...

=== OPTIMAL Q-LAW WEIGHTS ===
w_a: 98.6
w_f: 1.5
w_g: 1.5
w_h: 1.5
w_k: 1.5
Final cost: 100000000.00
